# FuturesExecutor reference copy

This is a copy of `ttbar_analysis_pipeline.ipynb` configured for **local `FuturesExecutor`**
(`USE_HQ=False`, `USE_DASK=False`). Use it as a baseline next to the HQ-wired notebook.

`maxchunks=1` is set on `Runner` so a smoke execution finishes in reasonable time over xrootd;
raise or remove it for a full AGC-sized run.


# CMS Open Data $t\bar{t}$: from data delivery to statistical inference

We are using [2015 CMS Open Data](https://cms.cern/news/first-cms-open-data-lhc-run-2-released) in this demonstration to showcase an analysis pipeline.
It features data delivery and processing, histogram construction and visualization, as well as statistical inference.

This notebook was developed in the context of the [IRIS-HEP AGC tools 2022 workshop](https://indico.cern.ch/e/agc-tools-2).
This work was supported by the U.S. National Science Foundation (NSF) Cooperative Agreement OAC-1836650 (IRIS-HEP).

This is a **technical demonstration**.
We are including the relevant workflow aspects that physicists need in their work, but we are not focusing on making every piece of the demonstration physically meaningful.
This concerns in particular systematic uncertainties: we capture the workflow, but the actual implementations are more complex in practice.
If you are interested in the physics side of analyzing top pair production, check out the latest results from [ATLAS](https://twiki.cern.ch/twiki/bin/view/AtlasPublic/TopPublicResults) and [CMS](https://cms-results.web.cern.ch/cms-results/public-results/preliminary-results/)!
If you would like to see more technical demonstrations, also check out an [ATLAS Open Data example](https://indico.cern.ch/event/1076231/contributions/4560405/) demonstrated previously.

This notebook implements most of the analysis pipeline shown in the following picture, using the tools also mentioned there:
![ecosystem visualization](utils/ecosystem.png)

### Data pipelines

There are two possible pipelines: one with `ServiceX` enabled, and one using only `coffea` for processing.
![processing pipelines](utils/processing_pipelines.png)

### Imports: setting up our environment

In [1]:
import logging
import time

import awkward as ak
import cabinetry
import cloudpickle
import correctionlib
from coffea import processor
from coffea.nanoevents import NanoAODSchema
from coffea.analysis_tools import PackedSelection
import copy
import hist
import matplotlib.pyplot as plt
import numpy as np
import pyhf

import utils  # contains code for bookkeeping and cosmetics, as well as some boilerplate

logging.getLogger("cabinetry").setLevel(logging.INFO)

### Configuration: number of files and data delivery path

The number of files per sample set here determines the size of the dataset we are processing. There are 9 samples being used here, all part of the 2015 CMS Open Data release.

These samples were originally published in miniAOD format, but for the purposes of this demonstration were pre-converted into nanoAOD format. More details about the inputs can be found [here](https://github.com/iris-hep/analysis-grand-challenge/tree/main/datasets/cms-open-data-2015).

The table below summarizes the amount of data processed depending on the `N_FILES_MAX_PER_SAMPLE` setting.

| setting | number of files | total size | number of events |
| --- | --- | --- | --- |
| `1` | 9 | 22.9 GB | 10,455,719 |
| `2` | 18 | 42.8 GB | 19,497,435 |
| `5` | 43 | 105 GB | 47,996,231 |
| `10` | 79 | 200 GB | 90,546,458 |
| `20` | 140 | 359 GB | 163,123,242 |
| `50` | 255 | 631 GB | 297,247,463 |
| `100` | 395 | 960 GB | 470,397,795 |
| `200` | 595 | 1.40 TB | 705,273,291 |
| `-1` | 787 | 1.78 TB | 940,160,174 |

The input files are all in the 1–3 GB range.

In [2]:
### GLOBAL CONFIGURATION
# input files per process, set to e.g. 10 (smaller number = faster)
# Use 1 for a small HQ vs Futures check (step 7).
N_FILES_MAX_PER_SAMPLE = 1

# enable Dask (False locally — coffea_casa TLS cluster is not available here)
USE_DASK = False

# enable HQ (CoffeaHQExecutor). Requires redis + TLS HQ server and HQ_RESULT_DIR.
# Mutually exclusive with USE_DASK for this notebook.
USE_HQ = False

# HQ connection (ignored unless USE_HQ)
HQ_HOST = "https://localhost"
HQ_PORT = 3000
HQ_VERIFY = "cert.pem"  # path relative to repo root, or absolute
HQ_N_WORKERS = 2

# enable ServiceX, specify options
USE_SERVICEX = False
USE_SERVICEX_UPROOT_RAW = True # set False to use func_adl instead

### ML-INFERENCE SETTINGS

# enable ML inference (False for faster HQ bring-up / missing models/)
USE_INFERENCE = False

# enable inference using NVIDIA Triton server
USE_TRITON = False


### Defining our `coffea` Processor

The processor includes a lot of the physics analysis details:
- event filtering and the calculation of observables,
- event weighting,
- calculating systematic uncertainties at the event and object level,
- filling all the information into histograms that get aggregated and ultimately returned to us by `coffea`.

#### Machine Learning Task

During the processing step, machine learning is used to calculate one of the variables used for this analysis. The models used are trained separately in the `jetassignment_training.ipynb` notebook. Jets in the events are assigned to labels corresponding with their parent partons using a boosted decision tree (BDT). More information about the model and training can be found within that notebook.

In [3]:
# Processor lives in ttbar_processor.py so the notebook and HQ vs Futures compare share one definition.
from ttbar_processor import TtbarAnalysis


### "Fileset" construction and metadata

Here, we gather all the required information about the files we want to process: paths to the files and asociated metadata.

In [4]:
fileset = utils.file_input.construct_fileset(
    N_FILES_MAX_PER_SAMPLE,
    use_xcache=False,
    af_name=utils.config["benchmarking"]["AF_NAME"],  # local files on /data for af_name="ssl-dev"
    input_from_eos=utils.config["benchmarking"]["INPUT_FROM_EOS"],
    xcache_atlas_prefix=utils.config["benchmarking"]["XCACHE_ATLAS_PREFIX"],
)

print(f"processes in fileset: {list(fileset.keys())}")
print(f"\nexample of information in fileset:\n{{\n  'files': [{fileset['ttbar__nominal']['files'][0]}, ...],")
print(f"  'metadata': {fileset['ttbar__nominal']['metadata']}\n}}")

processes in fileset: ['ttbar__nominal', 'ttbar__scaledown', 'ttbar__scaleup', 'ttbar__ME_var', 'ttbar__PS_var', 'single_top_s_chan__nominal', 'single_top_t_chan__nominal', 'single_top_tW__nominal', 'wjets__nominal']

example of information in fileset:
{
  'files': [https://xrootd-local.unl.edu:1094//store/user/AGC/nanoAOD/TT_TuneCUETP8M1_13TeV-powheg-pythia8/cmsopendata2015_ttbar_19980_PU25nsData2015v1_76X_mcRun2_asymptotic_v12_ext3-v1_00000_0000.root, ...],
  'metadata': {'process': 'ttbar', 'variation': 'nominal', 'nevts': 1334428, 'xsec': 729.84}
}


### ServiceX-specific functionality: query setup

Use one of two query languages (func_adl and uproot-raw) to define the query to be used for the purpose of extracting columns and filtering.

In [5]:
def get_query(source):
    """Query for event / column selection: >=4j >=1b, ==1 lep with pT>30 GeV + additional cuts,
    return relevant columns
    *NOTE* jet pT cut is set lower to account for systematic variations to jet pT
    """
    cuts = source.FromTree("Events")\
                 .Where(lambda e: {"pt": e.Electron_pt,
                               "eta": e.Electron_eta,
                               "cutBased": e.Electron_cutBased,
                               "sip3d": e.Electron_sip3d,}.Zip()\
                        .Where(lambda electron: (electron.pt > 30
                                                 and abs(electron.eta) < 2.1
                                                 and electron.cutBased == 4
                                                 and electron.sip3d < 4)).Count()
                        + {"pt": e.Muon_pt,
                           "eta": e.Muon_eta,
                           "tightId": e.Muon_tightId,
                           "sip3d": e.Muon_sip3d,
                           "pfRelIso04_all": e.Muon_pfRelIso04_all}.Zip()\
                        .Where(lambda muon: (muon.pt > 30
                                             and abs(muon.eta) < 2.1
                                             and muon.tightId
                                             and muon.pfRelIso04_all < 0.15)).Count()== 1)\
                        .Where(lambda f: {"pt": f.Jet_pt,
                                          "eta": f.Jet_eta,
                                          "jetId": f.Jet_jetId}.Zip()\
                               .Where(lambda jet: (jet.pt > 25
                                                   and abs(jet.eta) < 2.4
                                                   and jet.jetId == 6)).Count() >= 4)\
                        .Where(lambda g: {"pt": g.Jet_pt,
                                          "eta": g.Jet_eta,
                                          "btagCSVV2": g.Jet_btagCSVV2,
                                          "jetId": g.Jet_jetId}.Zip()\
                        .Where(lambda jet: (jet.btagCSVV2 > 0.5
                                            and jet.pt > 25
                                            and abs(jet.eta) < 2.4)
                                            and jet.jetId == 6).Count() >= 1)
    selection = cuts.Select(lambda h: {"Electron_pt": h.Electron_pt,
                                       "Electron_eta": h.Electron_eta,
                                       "Electron_phi": h.Electron_phi,
                                       "Electron_mass": h.Electron_mass,
                                       "Electron_cutBased": h.Electron_cutBased,
                                       "Electron_sip3d": h.Electron_sip3d,
                                       "Muon_pt": h.Muon_pt,
                                       "Muon_eta": h.Muon_eta,
                                       "Muon_phi": h.Muon_phi,
                                       "Muon_mass": h.Muon_mass,
                                       "Muon_tightId": h.Muon_tightId,
                                       "Muon_sip3d": h.Muon_sip3d,
                                       "Muon_pfRelIso04_all": h.Muon_pfRelIso04_all,
                                       "Jet_mass": h.Jet_mass,
                                       "Jet_pt": h.Jet_pt,
                                       "Jet_eta": h.Jet_eta,
                                       "Jet_phi": h.Jet_phi,
                                       "Jet_qgl": h.Jet_qgl,
                                       "Jet_btagCSVV2": h.Jet_btagCSVV2,
                                       "Jet_jetId": h.Jet_jetId,
                                       "event": h.event,
                                      })
    if USE_INFERENCE:
        return selection

    # some branches are only needed if USE_INFERENCE is turned on
    return selection.Select(lambda h: {"Electron_pt": h.Electron_pt,
                                       "Electron_eta": h.Electron_eta,
                                       "Electron_cutBased": h.Electron_cutBased,
                                       "Electron_sip3d": h.Electron_sip3d,
                                       "Muon_pt": h.Muon_pt,
                                       "Muon_eta": h.Muon_eta,
                                       "Muon_tightId": h.Muon_tightId,
                                       "Muon_sip3d": h.Muon_sip3d,
                                       "Muon_pfRelIso04_all": h.Muon_pfRelIso04_all,
                                       "Jet_mass": h.Jet_mass,
                                       "Jet_pt": h.Jet_pt,
                                       "Jet_eta": h.Jet_eta,
                                       "Jet_phi": h.Jet_phi,
                                       "Jet_btagCSVV2": h.Jet_btagCSVV2,
                                       "Jet_jetId": h.Jet_jetId,
                                      })

def get_uproot_raw_query():
    cut = '((count_nonzero((Electron_pt > 30) & (abs(Electron_eta) < 2.1) & (Electron_cutBased == 4) & (Electron_sip3d < 4), axis=1)' \
          '+ count_nonzero((Muon_pt > 30) & (abs(Muon_eta) < 2.1) & (Muon_tightId) & (Muon_pfRelIso04_all < 0.15), axis=1)) == 1)' \
          '& (count_nonzero((Jet_pt > 25) & (abs(Jet_eta) < 2.4) & (Jet_jetId == 6), axis=1) >= 4)' \
          '& (count_nonzero((Jet_pt > 25) & (abs(Jet_eta) < 2.4) & (Jet_jetId == 6) & (Jet_btagCSVV2 > 0.5), axis=1) >= 1)'
    branch_filter =  ['Electron_pt',
                      'Electron_eta',
                      'Electron_cutBased',
                      'Electron_sip3d',
                      'Muon_pt',
                      'Muon_eta',
                      'Muon_tightId',
                      'Muon_sip3d',
                      'Muon_pfRelIso04_all',
                      'Jet_mass',
                      'Jet_pt',
                      'Jet_eta',
                      'Jet_phi',
                      'Jet_qgl',
                      'Jet_btagCSVV2',
                      'Jet_jetId',
                     ]
    if USE_INFERENCE:
        branch_filter += [
                      'Electron_phi',
                      'Electron_mass',
                      'Muon_phi',
                      'Muon_mass',
                      'event',
        ]
    return query.UprootRaw({'treename': {'Events': 'servicex'}, 'cut': cut, 'filter_name': branch_filter})

### Caching the queried datasets with `ServiceX`

Using the queries created with `func_adl` or `uproot-raw`, we are using `ServiceX` to read the CMS Open Data files to build cached files with only the specific event information as dictated by the query.

In [6]:
if USE_SERVICEX:
    from servicex import deliver, query, dataset
    # dummy dataset on which to generate the query
    if USE_SERVICEX_UPROOT_RAW:
        hl_query = get_uproot_raw_query()
    else:
        hl_query = get_query(query.FuncADL_Uproot())

    # now we query the files using a wrapper around ServiceXDataset to transform all processes at once
    t0 = time.time()

    bundle = { 'Sample': [ { 'Name': _[0], 'Dataset': dataset.FileList(_[1]['files']),
                            'Query': hl_query,
                            'IgnoreLocalCache': utils.config["global"]["SERVICEX_IGNORE_CACHE"]
                           }
                           for _ in fileset.items() ] }
    if not utils.config["global"]["USE_SERVICEX_DOWNLOAD"]:
        bundle['General'] = { 'Delivery': 'URLs' }
    files_per_process = deliver(bundle)

    print(f"ServiceX data delivery took {time.time() - t0:.2f} seconds")

    # update fileset to point to ServiceX-transformed files
    for process in files_per_process.keys():
        fileset[process]["files"] = files_per_process[process]

### Execute the data delivery pipeline

What happens here depends on the flag `USE_SERVICEX`. If set to true, the processor is run on the data previously gathered by ServiceX, then will gather output histograms.

When `USE_SERVICEX` is false, the input files need to be processed during this step as well.

In [7]:
NanoAODSchema.warn_missing_crossrefs = False # silences warnings about branches we will not use here

if USE_DASK and USE_HQ:
    raise ValueError("Set only one of USE_DASK or USE_HQ")

if USE_DASK:
    cloudpickle.register_pickle_by_value(utils) # serialize methods and objects in utils so that they can be accessed within the coffea processor
    executor = processor.DaskExecutor(client=utils.clients.get_client(af=utils.config["global"]["AF"]))
elif USE_HQ:
    from pathlib import Path
    from hq.coffea import CoffeaHQExecutor

    cloudpickle.register_pickle_by_value(utils)
    verify = HQ_VERIFY
    if verify and not Path(verify).is_absolute():
        # cert.pem lives at repo root when the notebook cwd is example/
        candidate = Path("..") / verify
        verify = str(candidate.resolve() if candidate.exists() else Path(verify))
    executor = CoffeaHQExecutor(
        host=HQ_HOST,
        port=HQ_PORT,
        verify=verify,
        n_workers=HQ_N_WORKERS,
        pickle_modules=(utils,),
        poll_interval=1.0,
        status=True,
    )
else:
    executor = processor.FuturesExecutor(workers=utils.config["benchmarking"]["NUM_CORES"])

run = processor.Runner(
    executor=executor,
    schema=NanoAODSchema,
    savemetrics=True,
    metadata_cache={},
    chunksize=utils.config["benchmarking"]["CHUNKSIZE"],
    maxchunks=1,  # smoke: one chunk per file; remove for full run
)

if USE_SERVICEX:
    treename = "servicex"
else:
    treename = "Events"

# load local models if not using Triton or FuturesExecutor and models are not yet loaded
if USE_INFERENCE and not USE_TRITON and (USE_DASK or USE_HQ) and utils.ml.model_even is None and utils.ml.model_odd is None:
    utils.ml.load_models()

filemeta = run.preprocess(fileset, treename=treename)  # pre-processing

t0 = time.monotonic()
# processing
all_histograms, metrics = run(
    fileset,
    processor_instance=TtbarAnalysis(USE_INFERENCE, USE_TRITON),
    treename=treename,
)
exec_time = time.monotonic() - t0

print(f"\nexecution took {exec_time:.2f} seconds")

Output()

Output()

/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_genPartIdx => GenPart
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for FatJet_subJetIdx1 => SubJet
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-r

/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_genPartIdx => GenPart
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for FatJet_subJetIdx1 => SubJet
  warnings.warn(
/home/bothsides/anaconda3/envs/coffea_env/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-r


execution took 30.90 seconds


In [8]:
# track metrics
utils.metrics.track_metrics(metrics, fileset, exec_time, USE_DASK, USE_SERVICEX, N_FILES_MAX_PER_SAMPLE, USE_INFERENCE, USE_TRITON)

metrics saved as metrics/local-20260806-085251.json
event rate per worker (pure processtime): 20.29 kHz
amount of data read: 212.22 MB (note that this can be buggy: https://github.com/CoffeaTeam/coffea/issues/717)
